# 🌐 สัปดาห์ที่ 10: Interacting with Web APIs: Fetching and Processing JSON Data
### สำหรับนักศึกษาชั้นปีที่ 2 สาขาวิชาวิทยาการคอมพิวเตอร์ / เทคโนโลยีสารสนเทศ

สมุดงาน (Notebook) นี้จัดทำขึ้นเพื่อให้นักศึกษาสามารถเรียนรู้และทดลองปฏิบัติตามเนื้อหาบทเรียนได้ทันทีผ่าน Google Colab โดยไม่ต้องติดตั้งโปรแกรมบนเครื่องคอมพิวเตอร์

---
### 🎯 วัตถุประสงค์การเรียนรู้
1. เข้าใจหลักการทำงานของ **API (Application Programming Interface)** และ **REST API**
2. รู้วิธีติดตั้งและเรียกใช้งานไลบรารี `requests` ใน Python
3. สามารถส่งคำขอแบบ **HTTP GET Request** ไปยัง Server เพื่อดึงข้อมูลได้
4. สามารถแปลงและเข้าถึงข้อมูลในฟอร์แมต **JSON (JavaScript Object Notation)** ได้
5. มีทักษะการจัดการข้อผิดพลาด (**Error Handling**) เพื่อรองรับปัญหาจาก Network หรือ API Response

## 🛠️ ขั้นตอนที่ 1: ติดตั้งไลบรารี requests และเตรียมไดเรกทอรี

In [ ]:
# ติดตั้งไลบรารี requests ในสภาพแวดล้อม Colab
!pip install requests

# สร้างโฟลเดอร์สำหรับเก็บโค้ดโมดูล
!mkdir -p src
print("✅ สภาพแวดล้อมพร้อมใช้งานเรียบร้อยแล้ว!")

## 📦 ขั้นตอนที่ 2: สร้างโมดูล `src/api_client.py`
เราจะใช้คำสั่ง `%%writefile` เพื่อบันทึกโค้ดลงในไฟล์ `src/api_client.py`

In [ ]:
%%writefile src/api_client.py
import requests
import json
from typing import Any, Dict, List, Optional

class APIClient:
    """
    คลาสสำหรับเชื่อมต่อกับ JSONPlaceholder REST API
    """
    BASE_URL = "https://jsonplaceholder.typicode.com"

    def _make_request(self, endpoint: str, timeout: int = 10) -> Optional[Any]:
        url = f"{self.BASE_URL}{endpoint}"
        print(f"[INFO] กำลังส่งคำขอ (GET) ไปที่: {url}")
        try:
            response = requests.get(url, timeout=timeout)
            response.raise_for_status()  # แจ้งเตือนเมื่อสถานะเป็น 4xx หรือ 5xx
            return response.json()
        except requests.exceptions.HTTPError as errh:
            print(f"[ERROR] HTTP Error: {errh}")
        except requests.exceptions.ConnectionError as errc:
            print(f"[ERROR] Connection Error: {errc}")
        except requests.exceptions.Timeout as errt:
            print(f"[ERROR] Timeout Error: {errt}")
        except requests.exceptions.RequestException as err:
            print(f"[ERROR] Request Exception: {err}")
        except json.JSONDecodeError:
            print(f"[ERROR] ไม่สามารถแปลงข้อมูลเป็น JSON ได้: {response.text[:100]}...")
        return None

    def fetch_single_todo(self, todo_id: int) -> Optional[Dict[str, Any]]:
        return self._make_request(f"/todos/{todo_id}")

    def fetch_all_posts(self) -> Optional[List[Dict[str, Any]]]:
        return self._make_request("/posts")

    def fetch_user_todos(self, user_id: int) -> Optional[List[Dict[str, Any]]]:
        return self._make_request(f"/todos?userId={user_id}")


## 📄 ขั้นตอนที่ 3: สร้างไฟล์ `src/__init__.py` เพื่อกำหนดความเป็น Package

In [ ]:
%%writefile src/__init__.py
# File: src/__init__.py


## 🧪 ขั้นตอนที่ 4: ทดสอบการเรียกใช้งาน APIClient โดยตรงใน Notebook
ลองทดสอบดึงข้อมูลงานเดี่ยว (TODO) และแสดงรายการโพสต์ 3 รายการแรก

In [ ]:
import sys
import os

# เพิ่ม src ลงใน sys.path
sys.path.append(os.path.join(os.getcwd(), 'src'))
from api_client import APIClient

client = APIClient()

# 1. ทดสอบดึง TODO ID 1
todo = client.fetch_single_todo(1)
print("\n--- ตัวอย่างข้อมูล TODO 1 ---")
print(f"ID: {todo.get('id')}")
print(f"Title: {todo.get('title')}")
print(f"Completed: {todo.get('completed')}")

# 2. ทดสอบดึงรายการโพสต์ (แสดง 3 รายการ)
posts = client.fetch_all_posts()
print(f"\n--- ดึงข้อมูลโพสต์ได้ทั้งหมด {len(posts)} รายการ (แสดง 3 รายการแรก) ---")
for p in posts[:3]:
    print(f"[{p['id']}] {p['title']}")

## 🚀 ขั้นตอนที่ 5: พัฒนาส่วนขยายสำหรับการบ้าน (Assignment Tasks)
ทดลองเขียนฟังก์ชันดึงโปรไฟล์ผู้ใช้งาน และฟังก์ชันดึงคอมเมนต์ใต้โพสต์

In [ ]:
class ExtendedAPIClient(APIClient):
    def fetch_user_profile(self, user_id: int):
        """[Task 1] ดึงข้อมูลโปรไฟล์ผู้ใช้"""
        return self._make_request(f"/users/{user_id}")

    def fetch_post_comments(self, post_id: int):
        """[Task 2] ดึงคอมเมนต์ของโพสต์"""
        return self._make_request(f"/posts/{post_id}/comments")

ext_client = ExtendedAPIClient()

# ทดสอบดึงโปรไฟล์ User 1
user = ext_client.fetch_user_profile(1)
if user:
    print("\n--- ข้อมูล User Profile ---")
    print(f"ชื่อ: {user.get('name')}")
    print(f"อีเมล: {user.get('email')}")
    print(f"บริษัท: {user.get('company', {}).get('name')}")
    print(f"เมือง: {user.get('address', {}).get('city')}")

# ทดสอบดึงคอมเมนต์ของ Post 1
comments = ext_client.fetch_post_comments(1)
if comments:
    print(f"\n--- พบคอมเมนต์ทั้งหมด {len(comments)} รายการ (แสดงรายการแรก) ---")
    print(f"ผู้เขียน: {comments[0]['name']}")
    print(f"ข้อความ: {comments[0]['body']}")

## 💾 ขั้นตอนที่ 6: การบันทึกข้อมูลลงไฟล์ JSON (Export to JSON)
การจัดเก็บข้อมูลผลลัพธ์ลงเครื่องคอมพิวเตอร์

In [ ]:
import json

def export_to_json_file(filename: str, data: any) -> bool:
    try:
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4, ensure_ascii=False)
        print(f"✅ บันทึกไฟล์สำเร็จ: {filename}")
        return True
    except Exception as e:
        print(f"❌ เกิดข้อผิดพลาด: {e}")
        return False

# ทดลองบันทึกข้อมูล User ลงไฟล์
if user:
    export_to_json_file("user_1_colab.json", user)

# ตรวจสอบเนื้อหาไฟล์ที่บันทึก
!cat user_1_colab.json